In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [2]:
# Fetch all existing instance names matching tsm-sc-*
fetch_cmd = f'''
gcloud compute instances list \
    --filter="name~'tsm-sc-'" \
    --format="value(name)"
'''
instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

print("\n➡ Existing instances to delete:", instances_to_delete)

if instances_to_delete:
    # Use parallel deletion
    def delete_instance(instance_name):
        cmd = f'''
        gcloud compute instances delete {instance_name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"Deleting: {instance_name}")
        return subprocess.call(cmd, shell=True)

    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
        concurrent.futures.wait(futures)

    print("🧹 All old tsm-sc-* instances deleted.\n")
else:
    print("✔ No previous instances found to delete.\n")


➡ Existing instances to delete: []
✔ No previous instances found to delete.



[main e464d90] testing
 6 files changed, 1732 insertions(+), 340 deletions(-)
 create mode 100644 .ipynb_checkpoints/post-checkpoint.ipynb
 create mode 100644 latency.png
 create mode 100644 throughput.png


To github.com:tejas-shivanand-mane/stellar-core.git
   c3e6a2f..e464d90  main -> main


0

In [4]:
num_nodes = 4
project = "ucr-ursa-major-lesani-lab"
zone = "us-central1-c"
machine_type = "e2-highcpu-8"
image_name = "tsm-sc-image"  # your custom image
subnet = "default"
gcp_username = "tejas"

# Cleanup any existing instances with same prefix
os.system(f'gcloud compute instances delete --zone={zone} --quiet '
          f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')

# Create commands list
commands = []

for i in range(num_nodes):
    cmd = f'''
    gcloud compute instances create tsm-sc-{i:03} \
        --project={project} \
        --zone={zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=961693926925-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image={image_name},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''
    commands.append(cmd.strip())


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


# #Parallel instance creation

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(run_command, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

print("All instances launched.")

# Wait a bit for IPs to propagate
import time
time.sleep(30)

# Get IPs
os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
          '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')

with open('tsm_ips.txt', 'r') as f:
    iplist = [line.strip() for line in f.readlines()]

print("🎯 Instance IPs:", iplist)

ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highcpu-8         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image=tsm-sc-image,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-e

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].
Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-002  us-central1-c  e2-highcpu-8               10.128.0.41  34.46.121.194  RUNNING
NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-001  us-central1-c  e2-highcpu-8               10.128.0.50  35.224.64.145  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP      STATUS
tsm-sc-003  us-central1-c  e2-highcpu-8               10.128.0.52  136.116.216.226  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-000  us-central1-c  e2-highcpu-8               10.128.0.51  34.31.189.86  RUNNING
All instances launched.
🎯 Instance IPs: ['10.128.0.51', '10.128.0.50', '10.128.0.41', '10.128.0.52']


In [ ]:
os.system('git add .; git commit -m "testing "; git push')


In [5]:





def git_pull_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
print(results)

From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..e464d90  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..e464d90  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..e464d90  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..e464d90  main       -> origin/main


Updating e10985a..e464d90
Fast-forward
 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb | 1533 +++++++++++++++++++
 .ipynb_checkpoints/post-checkpoint.ipynb     |  153 ++
 GCP_setup.sh                                 |   46 +-
 SetupGCP.ipynb                               | 2121 ++++++++++++++++++++++++++
 gcp_setup_stellar_private.sh                 |  150 ++
 latency.png                                  |  Bin 0 -> 50809 bytes
 post.ipynb                                   |  153 ++
 src/overlay/OverlayManagerImpl.cpp           |  168 +-
 throughput.png                               |  Bin 0 -> 67185 bytes
 tsm_ips.txt                                  |    4 +
 10 files changed, 4201 insertions(+), 127 deletions(-)
 create mode 100644 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/post-checkpoint.ipynb
 create mode 100644 SetupGCP.ipynb
 create mode 100755 gcp_setup_stellar_private.sh
 create mode 100644 latency.png
 create mode 100644 post.ipynb
 cre

In [6]:
import shutil

if os.path.exists('../stellar-private'):
    
    shutil.rmtree('../stellar-private')
os.mkdir('../stellar-private')


os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')

os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')

Detected 4 nodes based on tsm_ips.txt.
Cleaning and creating directory for node1. Ports: Peer 11625, HTTP 11626...
Cleaning and creating directory for node2. Ports: Peer 11635, HTTP 11636...
Cleaning and creating directory for node3. Ports: Peer 11645, HTTP 11646...
Cleaning and creating directory for node4. Ports: Peer 11655, HTTP 11656...
Generating seed for node1...
Generating seed for node2...
Generating seed for node3...
Generating seed for node4...
Creating config file for node1...
Creating config file for node2...
Creating config file for node3...
Creating config file for node4...
Detected 4 nodes based on tsm_ips.txt.
Initializing database for node1...
Initializing database for node2...
Initializing database for node3...
Initializing database for node4...
✅ 4-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-co

2025-12-11T11:04:48.293 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2025-12-11T11:04:48.294 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node3", "node4", "GBJ74", "node2" ]
}

2025-12-11T11:04:48.294 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2025-12-11T11:04:48.318 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2025-12-11T11:04:48.318 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node3", "node4", "node1", "GCTLP" ]
}

2025-12-11T11:04:48.318 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2025-12-11T11:04:48.341 [default INFO] Config from /home/tejas/stellar-private/node3/stellar-core.cfg
2025-12-11T11:04:48.341 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "GAA2B", "node4", "node1", "node2" ]
}

2025-12-11T11:04:48.341 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2025-12-11T11:04:48.366 [default INFO] Config fro

0

In [7]:
# --- Configuration ---
line_to_add = "SEND_CUSTOM_MESSAGE=true"
target_file = "../stellar-private/node1/stellar-core.cfg" 

# --- The os.system() Command ---
# This command prepends the line to the target_file on your local machine.
os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')

print(f"The line '{line_to_add}' has been prepended to {target_file}.")

The line 'SEND_CUSTOM_MESSAGE=true' has been prepended to ../stellar-private/node1/stellar-core.cfg.


In [ ]:
def compile_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
make -j8; cd; sudo rm -r stellar-private"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(compile_stellar)(i) for i in range(len(iplist)))
print(results)

make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make  all-recursive
make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home

/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)


make[5]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium'
Making all in include
make[6]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium/include'
make[6]: Nothing to be done for 'all'.
make[6]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium/include'
make[6]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium'
make[6]: Nothing to be done for 'all-am'.
make[6]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium'
make[5]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium'
make[5]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src'
make[5]: Nothing to be done for 'all-am'.
make[5]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src'
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src'
Making all in test
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/test'
Making all in default
make[3]: Ent

/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)


make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[2]: Entering directory '/home/tejas/stellar-core/src'
echo '#include "main/StellarCoreVersion.h"

const std::string STELLAR_CORE_VERSION = "e464d90";' > main/StellarCoreVersion.cpp
make  all-am
make[2]: Entering directory '/home/tejas/stellar-core/src'
echo '#include "main/StellarCoreVersion.h"

const std::string STELLAR_CORE_VERSION = "e464d90";' > main/StellarCoreVersion.cpp
make  all-am
make[2]: Entering directory '/home/tejas/stellar-core/src'
echo '#include "main/StellarCoreVersion.h"

const std::string STELLAR_CORE_VERSION = "e464d90";' > main/StellarCoreVersion.cpp
make  all-am
make[3]: Entering directory '/home/tejas/stellar-core/src'
depbase=`echo overlay/OverlayManagerImpl.o | sed 's|[^/]*$|.deps/&|;s|\.o$||'`;\
g++ -std=c++17 -DHAVE_CONFIG

overlay/OverlayManagerImpl.cpp: In function ‘void submitAccountCreationTransaction(stellar::Application&)’:
overlay/OverlayManagerImpl.cpp:139:59: warning: converting to ‘stellar::PublicKey’ from initializer list would use explicit constructor ‘stellar::PublicKey::PublicKey(stellar::PublicKeyType)’
  139 |             le.data.account().inflationDest.activate() = {};
      |                                                           ^
In file included from ../src/protocol-curr/xdr/Stellar-contract.h:10,
                 from ./overlay/StellarXDR.h:2,
                 from ./database/Database.h:9,
                 from ./overlay/Peer.h:8,
                 from ./overlay/OverlayManagerImpl.h:7,
                 from overlay/OverlayManagerImpl.cpp:5:
../src/protocol-curr/xdr/Stellar-types.h:298:12: note: ‘stellar::PublicKey::PublicKey(stellar::PublicKeyType)’ declared here
  298 |   explicit PublicKey(PublicKeyType which = PublicKeyType{}) : type_(which) {
      |            ^~~~~~~~~
overl

In [ ]:
def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
    """
    Constructs and executes the gcloud compute scp command to copy a folder
    to a specific GCP instance.
    """
    instance_name = f"tsm-sc-{i:03}"
    
    # The --recurse flag is crucial for copying folders
    # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
    command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'

    print(f"Executing command for {instance_name}: {command}")
    
    # os.system executes the command and returns the exit status (0 for success)
    output = os.system(command)
    
    print(f"Command for {instance_name} finished with exit code: {output}")
    
    return (instance_name, output)


results = Parallel(n_jobs=20)(
    delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
)

In [ ]:
def setup_stellar_private(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd /home/tejas; \
cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
./gcp_setup_stellar_private.sh;"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
print(results)

In [ ]:
def run_stellar_private(i):
    # Calculate the node number (assuming i starts at 0, node starts at 1)
    node_number = i + 1 
    instance_name = f"tsm-sc-{i:03}"
    
    # ----------------------------------------------------------------------------------
    # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
    # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
    # '< /dev/null' ensures the process doesn't wait for input.
    # ----------------------------------------------------------------------------------
    remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""
    
    # Construct the full gcloud command
    command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
    
    print(f"Executing: {command}")
    
    # os.system should now return immediately because the remote shell exits
    output = os.system(command)
    print(f"Return code for {instance_name}: {output}")

# Corrected Loop (to run 0, 1, 2, 3)
# results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in range(1, num_nodes))
results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in range(num_nodes))

# run_stellar_private(0)
print(results)
print("All SSH commands executed. Nodes should be starting up in the background.")

In [ ]:
time.sleep(190)

In [ ]:


def kill_stellar_private(i):
    remote_command = f"""\
cd /home/tejas/stellar-private; \
sudo pkill stellar-core; \
"""
    
    # Construct the full gcloud command
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
    
    print(f"Executing: {command}")

    output = os.system(command)
    print(f"Return code for tsm-sc-{i:03}: {output}")


results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))


In [ ]:
remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "cp_" + str(num_nodes) 

# Ensure the local base destination directory exists
os.makedirs(local_base_destination, exist_ok=True)


def copy_folder_from_instance(i):
    """
    Constructs and executes the gcloud compute scp command to copy a specific 
    nodeN folder from instance i to a local folder named after the instance.
    """
    instance_name = f"tsm-sc-{i:03}"
    
    # Calculate the node number (assuming i starts at 0, node starts at 1)
    node_number = i + 1 
    node_folder = f"node{node_number}"

    # 1. Define the specific REMOTE source path on the instance
    # Example: /home/tejas/stellar-private/node1
    remote_source_path = os.path.join(remote_base_folder, node_folder)
    
    # 2. Define the LOCAL destination path
    # We'll use the instance name for the subfolder to keep backups separate
    local_destination_path = os.path.join(local_base_destination, instance_name)
    os.makedirs(local_destination_path, exist_ok=True)
    
    # The SCp command requires the remote path to be formatted as:
    # [INSTANCE_NAME]:[REMOTE_SRC]
    remote_source = f"{instance_name}:{remote_source_path}"
    
    # The command reverses the source (remote) and destination (local)
    command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'

    print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
    
    # os.system executes the command and returns the exit status (0 for success)
    output = os.system(command)
    
    print(f"Copy from {instance_name} finished with exit code: {output}")
    
    return (instance_name, output)

# ---
# Execute the copy operation in parallel
# ---

results = Parallel(n_jobs=20)(
    delayed(copy_folder_from_instance)(i) for i in range(num_nodes)
)

print("\n--- Summary of Download Results ---")
print(results)

In [ ]:

# # Fetch all existing instance names matching tsm-sc-*
# fetch_cmd = f'''
# gcloud compute instances list \
#     --filter="name~'tsm-sc-'" \
#     --format="value(name)"
# '''
# instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
# instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

# print("\n➡ Existing instances to delete:", instances_to_delete)

# if instances_to_delete:
#     # Use parallel deletion
#     def delete_instance(instance_name):
#         cmd = f'''
#         gcloud compute instances delete {instance_name} \
#             --zone={zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"Deleting: {instance_name}")
#         return subprocess.call(cmd, shell=True)

#     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
#         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
#         concurrent.futures.wait(futures)

#     print("🧹 All old tsm-sc-* instances deleted.\n")
# else:
#     print("✔ No previous instances found to delete.\n")
